In [1]:
# ── Konfiguracja 1: Wczytanie pakietów ──────────────────────

import pandas as pd                         # Praca na tabelach
from pathlib import Path                    # Obsługa projektu
import plotly.graph_objects as go           # Wizualizacja danych
from plotly.subplots import make_subplots   # Łączenie wykresów
import numpy as np                          # Obliczenia liczbowe

In [2]:
# ── Konfiguracja 2: Wczytanie danych z 01_etl ──────────────────────

# Konfiguracja ścieżek (zgodna z pierwszym notebookiem)
PROJECT_ROOT = Path.cwd().parent  # ponieważ notebook jest w notebooks/
DATA_DIR = PROJECT_ROOT / "processed"  # uwaga: bez "data" w środku

OUT_DIR = PROJECT_ROOT / "outputs" / "html"
OUT_DIR.mkdir(parents=True, exist_ok=True)

# Odczyt danych
df = pd.read_parquet(DATA_DIR / "df.parquet")
df_rok = pd.read_parquet(DATA_DIR / "df_rok.parquet")

In [3]:
# ── SEKCJA 1: DASHBOARD LINIOWY 3×2 (dane roczne z df) ──────────────────────


# Konwersja indeksu kwartalnego na PeriodIndex żeby resample działał
df.index = pd.PeriodIndex(df.index, freq='Q')

df_rok = df[['vat_mld', 'cit_mld', 'akcyza_mld', 'pkb_mld']].resample('Y').sum()
df_rok.index = df_rok.index.year   # indeks jako int roku (1999, 2000, ...)

# Zmienne pochodne na danych rocznych
df_rok['vat_pkb_pct']    = df_rok['vat_mld'] / df_rok['pkb_mld'] * 100
df_rok['vat_udzial_pct'] = (
    df_rok['vat_mld'] /
    (df_rok['vat_mld'] + df_rok['cit_mld'] + df_rok['akcyza_mld']) * 100
)
df_rok['vat_rr']         = df_rok['vat_mld'].pct_change() * 100
df_rok['cit_akc_mld']    = df_rok['cit_mld'] + df_rok['akcyza_mld']
df_rok['trend']          = range(1, len(df_rok) + 1)

# Przywróć indeks df do string (żeby nie psuć dalszych analiz)
df.index = df.index.astype(str)

print(f'✅ Agregacja roczna: {df_rok.index.min()}–{df_rok.index.max()}, '
      f'n={len(df_rok)}')
print(df_rok[['vat_mld', 'pkb_mld', 'vat_pkb_pct',
              'vat_udzial_pct']].tail(5).to_string())

# ══════════════════════════════════════════════════════════════════════════════
# DANE DLA DASHBOARDU
# ══════════════════════════════════════════════════════════════════════════════

lata = df_rok.index.tolist()

# Podział przed/po — rok 2016 wspólny punkt łączący segmenty
lata_przed = [r for r in lata if r <= 2016]
lata_po    = [r for r in lata if r >= 2016]

def split(kolumna):
    """Dzieli serię na dwa okresy. Rok 2016 wspólny — zapewnia ciągłość linii."""
    dane = df_rok[kolumna].round(2)
    return (
        dane[dane.index <= 2016].tolist(),
        dane[dane.index >= 2016].tolist(),
    )

# Indeks 1999=100
vat_n = (df_rok['vat_mld']    / df_rok['vat_mld'].iloc[0]    * 100).round(2).tolist()
cit_n = (df_rok['cit_mld']    / df_rok['cit_mld'].iloc[0]    * 100).round(2).tolist()
akc_n = (df_rok['akcyza_mld'] / df_rok['akcyza_mld'].iloc[0] * 100).round(2).tolist()

idx_2016 = lata.index(2016)
vat_n_przed, vat_n_po = vat_n[:idx_2016 + 1], vat_n[idx_2016:]
cit_n_przed, cit_n_po = cit_n[:idx_2016 + 1], cit_n[idx_2016:]
akc_n_przed, akc_n_po = akc_n[:idx_2016 + 1], akc_n[idx_2016:]

# Zmiana r/r
rr_vals = df_rok['vat_rr'].fillna(0).round(2).tolist()

# Średnie VAT/PKB przed/po
sr_przed = df_rok.loc[df_rok.index <= 2015, 'vat_pkb_pct'].mean()
sr_po    = df_rok.loc[df_rok.index >= 2016, 'vat_pkb_pct'].mean()

# Średnie udziału VAT przed/po
sr_ud_przed = df_rok.loc[df_rok.index <= 2015, 'vat_udzial_pct'].mean()
sr_ud_po    = df_rok.loc[df_rok.index >= 2016, 'vat_udzial_pct'].mean()

# ── KOLORY ────────────────────────────────────────────────────────────────────
CZERWONY    = '#C0392B'
NIEBIESKI   = '#1B3A6B'
ZLOTY       = '#C9982A'
TURKUSOWY   = '#1A7A6E'
ZIELONY     = '#1E8449'
MGLA        = '#8395A7'
SZARY_PRZED = '#A9B8C8'
SZARY_GRID  = '#E4EAF2'
BG          = '#F7F9FC'

FILL = {
    CZERWONY:  'rgba(192,57,43,0.07)',
    NIEBIESKI: 'rgba(27,58,107,0.07)',
    ZLOTY:     'rgba(201,152,42,0.07)',
    TURKUSOWY: 'rgba(26,122,110,0.07)',
    ZIELONY:   'rgba(30,132,73,0.07)',
}

# ── FIGURA 3×2 ────────────────────────────────────────────────────────────────
fig = make_subplots(
    rows=3, cols=2,
    subplot_titles=[
        '① VAT (mld PLN)',
        '④ Udział VAT w VAT+CIT+Akcyza (%)',
        '② VAT / PKB (%)',
        '⑤ Zmiana VAT r/r (%)',
        '③ PKB (mld PLN)',
        '⑥ CIT + Akcyza (mld PLN)',
    ],
    vertical_spacing=0.18,
    horizontal_spacing=0.10,
)

# ── HELPER: dwa segmenty dla jednej serii ─────────────────────────────────────
def dodaj_serie(fig, row, col, kolumna, kolor, hover_suffix, fill=False):
    """
    Rysuje serię jako dwa segmenty:
      przed 2016 — szary, kreskowany
      po 2016    — pełny kolor
    Rok 2016 wspólny — linia ciągła wizualnie.
    """
    dane_przed, dane_po = split(kolumna)

    fig.add_trace(go.Scatter(
        x=lata_przed, y=dane_przed,
        mode='lines+markers',
        line=dict(color=SZARY_PRZED, width=2.3, dash='dash'),
        marker=dict(size=5, color=SZARY_PRZED,
                    line=dict(color='white', width=0.8)),
        opacity=0.7, showlegend=False,
        hovertemplate=f'<b>%{{x}}</b><br>%{{y:.2f}} {hover_suffix}'
                      f'<extra>przed 2016</extra>',
    ), row=row, col=col)

    fig.add_trace(go.Scatter(
        x=lata_po, y=dane_po,
        mode='lines+markers',
        line=dict(color=kolor, width=2.5),
        marker=dict(size=5, color=kolor,
                    line=dict(color='white', width=0.8)),
        fill='tozeroy' if fill else 'none',
        fillcolor=FILL.get(kolor) if fill else None,
        showlegend=False,
        hovertemplate=f'<b>%{{x}}</b><br>%{{y:.2f}} {hover_suffix}'
                      f'<extra>po 2016</extra>',
    ), row=row, col=col)


# ── ① VAT — row1, col1 ───────────────────────────────────────────────────────
dodaj_serie(fig, 1, 1, 'vat_mld', CZERWONY, 'mld PLN', fill=True)

# ── ② VAT/PKB — row2, col1 ───────────────────────────────────────────────────
dodaj_serie(fig, 2, 1, 'vat_pkb_pct', NIEBIESKI, '%', fill=True)

# Linie średnich
fig.add_trace(go.Scatter(
    x=[lata[0], lata[-1]], y=[sr_przed, sr_przed],
    mode='lines', showlegend=False,
    line=dict(color=MGLA, width=1.5, dash='dot'),
    hovertemplate=f'Śr. 1999–2015: {sr_przed:.2f}%<extra></extra>',
), row=2, col=1)

fig.add_trace(go.Scatter(
    x=[lata[0], lata[-1]], y=[sr_po, sr_po],
    mode='lines', showlegend=False,
    line=dict(color=TURKUSOWY, width=1.5, dash='dot'),
    hovertemplate=f'Śr. 2016–2025: {sr_po:.2f}%<extra></extra>',
), row=2, col=1)

# ── ③ PKB — row3, col1 ───────────────────────────────────────────────────────
dodaj_serie(fig, 3, 1, 'pkb_mld', ZLOTY, 'mld PLN', fill=True)

# ── ④ UDZIAŁ VAT — row1, col2 ─────────────────────────────────────────────────
dodaj_serie(fig, 1, 2, 'vat_udzial_pct', ZIELONY, '%', fill=True)

# Linie średnich udziału
fig.add_trace(go.Scatter(
    x=[lata[0], lata[-1]], y=[sr_ud_przed, sr_ud_przed],
    mode='lines', showlegend=False,
    line=dict(color=MGLA, width=1.5, dash='dot'),
    hovertemplate=f'Śr. 1999–2015: {sr_ud_przed:.2f}%<extra></extra>',
), row=1, col=2)

fig.add_trace(go.Scatter(
    x=[lata[0], lata[-1]], y=[sr_ud_po, sr_ud_po],
    mode='lines', showlegend=False,
    line=dict(color=ZIELONY, width=1.5, dash='dot'),
    hovertemplate=f'Śr. 2016–2025: {sr_ud_po:.2f}%<extra></extra>',
), row=1, col=2)

# ── ⑤ VAT r/r — row2, col2 ───────────────────────────────────────────────────
kolory_bar = [
    SZARY_PRZED if r < 2016 else (CZERWONY if v >= 0 else MGLA)
    for r, v in zip(lata, rr_vals)
]

fig.add_trace(go.Bar(
    x=lata, y=rr_vals,
    marker_color=kolory_bar,
    marker_line_color='white',
    marker_line_width=0.5,
    opacity=0.85,
    showlegend=False,
    hovertemplate='<b>%{x}</b><br>Zmiana: %{y:.2f}%<extra></extra>',
), row=2, col=2)

# ── ⑥ CIT + AKCYZA — row3, col2 ──────────────────────────────────────────────
dodaj_serie(fig, 3, 2, 'cit_akc_mld', TURKUSOWY, 'mld PLN', fill=True)

# ── LINIE REFORM ─────────────────────────────────────────────────────────────
subplot_axes = [
    ('x',  'y'),
    ('x2', 'y2'),
    ('x3', 'y3'),
    ('x4', 'y4'),
    ('x5', 'y5'),
    ('x6', 'y6'),
]

REFORMY = [
    (2016, NIEBIESKI, 'JPK (2016)'),
    (2018, ZLOTY,     'Split payment (2018)'),
    (2019, ZIELONY,   'Biała lista (2019)'),
    (2020, MGLA,      'COVID-19 (2020)'),
]

shapes = []
for xref, yref in subplot_axes:
    for rok, kolor, _ in REFORMY:
        shapes.append(dict(
            type='line',
            xref=xref, yref=f'{yref} domain',
            x0=rok, x1=rok, y0=0, y1=1,
            line=dict(color=kolor, width=1.3, dash='dash'),
            opacity=0.65, layer='below',
        ))

# ── LEGENDA ───────────────────────────────────────────────────────────────────
for i, (rok, kolor, label) in enumerate(REFORMY):
    fig.add_trace(go.Scatter(
        x=[None], y=[None], mode='lines',
        name=label,
        legendgroup='reformy',
        legendgrouptitle_text='Reformy podatkowe' if i == 0 else None,
        line=dict(color=kolor, width=2, dash='dash'),
        showlegend=True,
    ))

for i, (kolor, dash, opacity, label) in enumerate([
    (SZARY_PRZED, 'dash',  0.7, 'Przed reformami (1999–2015)'),
    (CZERWONY,    'solid', 1.0, 'Po reformach (2016–2025)'),
]):
    fig.add_trace(go.Scatter(
        x=[None], y=[None], mode='lines',
        name=label,
        legendgroup='okresy',
        legendgrouptitle_text='Okres' if i == 0 else None,
        line=dict(color=kolor, width=2.5, dash=dash),
        opacity=opacity, showlegend=True,
    ))

# ── LAYOUT ───────────────────────────────────────────────────────────────────
fig.update_layout(
    title=dict(
        text=(
            '<b>EDA · Uszczelnienie VAT a dochody budżetowe Polski 1999–2025</b><br>'
            '<span style="font-size:11px;color:#8395A7">'
            'Źródła: Ministerstwo Finansów (Sprawozdania operatywne) · '
            'Eurostat (namq_10_gdp/CP_MNAC)<br>'
            'Opracowanie: Mateusz Durski'
            '</span>'
        ),
        x=0.01, xanchor='left',
        font=dict(size=15),
    ),
    height=1050,
    paper_bgcolor=BG,
    hovermode='x unified',
    template='simple_white',
    shapes=shapes,
    annotations=[],
    legend=dict(
        orientation='h',
        y=-0.07, x=0.5, xanchor='center',
        font=dict(size=10),
        bgcolor='rgba(247,249,252,0.9)',
        bordercolor=SZARY_GRID, borderwidth=1,
        tracegroupgap=30,
    ),
    margin=dict(t=110, b=100, l=60, r=60),
)

# ── OSIE ─────────────────────────────────────────────────────────────────────
etykiety_osi = [
    (1, 1, 'Rok', 'mld PLN'),
    (1, 2, 'Rok', '% łącznych wpływów'),
    (2, 1, 'Rok', '% PKB'),
    (2, 2, 'Rok', '% zmiana r/r'),
    (3, 1, 'Rok', 'mld PLN'),
    (3, 2, 'Rok', 'mld PLN'),
]
for row, col, xlabel, ylabel in etykiety_osi:
    fig.update_xaxes(
        title_text=xlabel, showgrid=False,
        tickangle=-45, dtick=2,   # co 2 lata — czytelne przy n=26
        row=row, col=col,
    )
    fig.update_yaxes(
        title_text=ylabel, showgrid=True,
        gridcolor=SZARY_GRID, row=row, col=col,
    )

# Grid fix
for axis in fig.layout:
    if 'xaxis' in axis:
        fig.layout[axis].update(showgrid=False)
    if 'yaxis' in axis:
        fig.layout[axis].update(showgrid=True, gridcolor=SZARY_GRID)

# ── EXPORT ───────────────────────────────────────────────────────────────────
file_path = OUT_DIR / "eda_vat_interaktywny.html"

fig.write_html(
    file_path,
    include_plotlyjs="cdn",
)

print(f"✅ Zapisano: {file_path}")

fig.show()


✅ Agregacja roczna: 1999–2025, n=27
            vat_mld    pkb_mld  vat_pkb_pct  vat_udzial_pct
kwartal                                                    
2021     215.733975  2660.9998     8.107253       62.730549
2022     230.390544  3095.7485     7.442160       60.581170
2023     244.267386  3411.4104     7.160305       61.536166
2024     287.683369  3658.8406     7.862692       65.645431
2025     321.646382  3904.4108     8.238026       67.310827
✅ Zapisano: c:\Users\duros\OneDrive\Pulpit\Projekty Python\VAT\projekt_vat\outputs\html\eda_vat_interaktywny.html


In [4]:
del akc_n, akc_n_po, akc_n_przed, axis, BG, cit_n, cit_n_po, cit_n_przed, col, CZERWONY, dash, etykiety_osi, fig, FILL, i, idx_2016, kolor, kolory_bar, label, lata, lata_po, lata_przed, MGLA, NIEBIESKI, opacity, REFORMY, rok, row, rr_vals, shapes, sr_po, sr_przed, sr_ud_po, sr_ud_przed, subplot_axes, SZARY_GRID, SZARY_PRZED, TURKUSOWY, vat_n, vat_n_po, vat_n_przed, xlabel, xref, ylabel, yref, ZIELONY, ZLOTY

In [5]:
# ── SEKCJA 2: BOXPLOT z podziałem przed i po reformach (dane kwartalne) ──────────────────────

# ── DANE — bez round(), oryginalne wartości z df ─────────────────────────────
vat_pkb_przed = df.loc[df.index <= '2016Q2', 'vat_pkb_pct'].tolist()
vat_pkb_po    = df.loc[df.index >= '2016Q3', 'vat_pkb_pct'].tolist()
vat_mld_przed = df.loc[df.index <= '2016Q2', 'vat_mld'].tolist()
vat_mld_po    = df.loc[df.index >= '2016Q3', 'vat_mld'].tolist()

# ── KOLORY — spójne z EDA ─────────────────────────────────────────────────────
SZARY_PRZED = '#A9B8C8'
CZERWONY    = '#C0392B'
NIEBIESKI   = '#1B3A6B'
SZARY_GRID  = '#E4EAF2'
BG          = '#F7F9FC'

# ── FIGURA ────────────────────────────────────────────────────────────────────
box = make_subplots(
    rows=1, cols=2,
    subplot_titles=(
        '① VAT / PKB (%) — efektywność poboru',
        '② VAT (mld PLN) — nominalne wpływy',
    ), vertical_spacing=0.01
)

# ── HELPER ────────────────────────────────────────────────────────────────────
def dodaj_box(fig, row, col, dane, nazwa, kolor, fill_alpha,
              showlegend, legendgroup, decimals=2):
    r, g, b = int(kolor[1:3], 16), int(kolor[3:5], 16), int(kolor[5:7], 16)
    fig.add_trace(go.Box(
        y=[round(v, decimals) for v in dane],   # zaokrąglenie tylko dla wykresu
        name=nazwa,
        marker=dict(
            color=kolor,
            size=6,
            line=dict(color='white', width=0.8),
            opacity=0.85,
        ),
        line=dict(color=kolor, width=1.8),
        fillcolor=f'rgba({r},{g},{b},{fill_alpha})',
        boxpoints='all',
        jitter=0.25,
        pointpos=0,
        legendgroup=legendgroup,
        showlegend=showlegend,
        hovertemplate=f'<b>{nazwa}</b><br>Wartość: %{{y:.{decimals}f}}<extra></extra>',
    ), row=row, col=col)


# ── PANEL 1: VAT/PKB ─────────────────────────────────────────────────────────
dodaj_box(box, 1, 1, vat_pkb_przed,
          nazwa='Przed 2016-Q3', kolor=SZARY_PRZED, fill_alpha=0.25,
          showlegend=True, legendgroup='przed', decimals=2)

dodaj_box(box, 1, 1, vat_pkb_po,
          nazwa='Po 2016-Q3', kolor=NIEBIESKI, fill_alpha=0.25,
          showlegend=True, legendgroup='po', decimals=2)

# ── PANEL 2: VAT mld ─────────────────────────────────────────────────────────
dodaj_box(box, 1, 2, vat_mld_przed,
          nazwa='Przed 2016-Q3', kolor=SZARY_PRZED, fill_alpha=0.25,
          showlegend=False, legendgroup='przed', decimals=1)

dodaj_box(box, 1, 2, vat_mld_po,
          nazwa='Po 2016-Q3', kolor=CZERWONY, fill_alpha=0.25,
          showlegend=False, legendgroup='po', decimals=1)

# ── ADNOTACJE — różnica średnich ──────────────────────────────────────────────
sr_pkb_przed = np.mean(vat_pkb_przed)
sr_pkb_po    = np.mean(vat_pkb_po)
sr_mld_przed = np.mean(vat_mld_przed)
sr_mld_po    = np.mean(vat_mld_po)

# ── LAYOUT ───────────────────────────────────────────────────────────────────
box.update_layout(
    title=dict(
        text=(
            '<b>Boxplot · Porównanie wpływów VAT przed i po reformach 2016-Q3</b><br>'
            '<span style="font-size:11px;color:#8395A7">'
            'Źródła: Ministerstwo Finansów (Sprawozdania operatywne) · '
            'Eurostat (namq_10_gdp/CP_MNAC)<br>'
            'Opracowanie: Mateusz Durski'
            '</span>'
        ),
        x=0.05, y=0.92,
        xanchor='left',
        font=dict(size=15),
    ),

    height=550,
    paper_bgcolor=BG,
    template='simple_white',

    showlegend=True,

    legend=dict(
        orientation='h',
        y=-0.15,
        x=0.5,
        xanchor='center',
        font=dict(size=10),
        bgcolor='rgba(247,249,252,0.9)',
        bordercolor=SZARY_GRID,
        borderwidth=1,
    ),

    margin=dict(
        t=120,
        b=80,
        l=60,
        r=60,
    ),
)

box.add_annotation(
    x=0.015,
    y=0.96,
    xref='paper',
    yref='paper',
    showarrow=False,
    align='center',
    font=dict(size=10, color=NIEBIESKI),
    text=(
        f'Śr. przed: <b>{sr_pkb_przed:.2f}%</b> → '
        f'Śr. po: <b>{sr_pkb_po:.2f}%</b><br>'
        f'Δ = <b>{sr_pkb_po - sr_pkb_przed:+.2f} pp</b>'
    )
)

box.add_annotation(
    x=0.64,
    y=0.96,
    xref='paper',
    yref='paper',
    showarrow=False,
    align='center',
    font=dict(size=10, color=CZERWONY),
    text=(
        f'Śr. przed: <b>{sr_mld_przed:.1f} mld</b> → '
        f'Śr. po: <b>{sr_mld_po:.1f} mld</b><br>'
        f'Δ = <b>{sr_mld_po - sr_mld_przed:+.1f} mld</b>'
    )
)

# ── OSIE ─────────────────────────────────────────────────────────────────────
box.update_yaxes(title_text='% PKB',   showgrid=True,
                 gridcolor=SZARY_GRID, row=1, col=1)
box.update_yaxes(title_text='mld PLN', showgrid=True,
                 gridcolor=SZARY_GRID, row=1, col=2)
box.update_xaxes(title_text='Okres', row=1, col=1)
box.update_xaxes(title_text='Okres', row=1, col=2)

# ── EXPORT ───────────────────────────────────────────────────────────────────
file_path = OUT_DIR / "boxplot.html"

box.write_html(
    file_path,
    include_plotlyjs="cdn",
)

print(f"✅ Zapisano: {file_path}")

box.show()

✅ Zapisano: c:\Users\duros\OneDrive\Pulpit\Projekty Python\VAT\projekt_vat\outputs\html\boxplot.html


In [6]:
del BG, box,CZERWONY, NIEBIESKI, sr_mld_po, sr_mld_przed, sr_pkb_po, sr_pkb_przed, SZARY_GRID, SZARY_PRZED, vat_mld_po, vat_mld_przed, vat_pkb_po, vat_pkb_przed

In [7]:
# ── SEKCJA 3: STATYSTYKI OPISOWE (dane kwartalne) ─────────────────────────────

# ── DANE ──────────────────────────────────────────────────────────────────────
ZMIENNE = [
    ('VAT (mld PLN)',     'vat_mld',     1),
    ('VAT / PKB (%)',     'vat_pkb_pct', 2),
    ('CIT (mld PLN)',     'cit_mld',     1),
    ('Akcyza (mld PLN)', 'akcyza_mld',   1),
]

# ── KOLORY ────────────────────────────────────────────────────────────────────
CZERWONY   = '#C0392B'
NIEBIESKI  = '#1B3A6B'
TURKUSOWY  = '#1A7A6E'
SZARY      = '#8395A7'
SZARY_GRID = '#E4EAF2'
BG         = '#F7F9FC'
BIALY      = '#FFFFFF'

# ── OBLICZENIA STATYSTYK ──────────────────────────────────────────────────────
def statystyki(kolumna, decimals):
    przed = df.loc[df.index <= '2016Q2', kolumna]
    po    = df.loc[df.index >  '2016Q3', kolumna]
    delta = po.mean() - przed.mean()
    return {
        'sr_przed':  round(przed.mean(),   decimals),
        'sr_po':     round(po.mean(),      decimals),
        'delta':     round(delta,          decimals),
        'med_przed': round(przed.median(), decimals),
        'med_po':    round(po.median(),    decimals),
        'std_przed': round(przed.std(),    decimals),
        'std_po':    round(po.std(),       decimals),
        'min_przed': round(przed.min(),    decimals),
        'min_po':    round(po.min(),       decimals),
        'max_przed': round(przed.max(),    decimals),
        'max_po':    round(po.max(),       decimals),
    }

# Budowa wierszy
wiersze = []
for nazwa, kolumna, decimals in ZMIENNE:
    s = statystyki(kolumna, decimals)
    wiersze.append({'nazwa': nazwa, 'decimals': decimals, **s})

# ── KOLUMNY ───────────────────────────────────────────────────────────────────
col_nazwa     = [w['nazwa']     for w in wiersze]
col_sr_przed  = [w['sr_przed']  for w in wiersze]
col_sr_po     = [w['sr_po']     for w in wiersze]
col_delta     = [w['delta']     for w in wiersze]
col_med_przed = [w['med_przed'] for w in wiersze]
col_med_po    = [w['med_po']    for w in wiersze]
col_std_przed = [w['std_przed'] for w in wiersze]
col_std_po    = [w['std_po']    for w in wiersze]
col_min_przed = [w['min_przed'] for w in wiersze]
col_min_po    = [w['min_po']    for w in wiersze]
col_max_przed = [w['max_przed'] for w in wiersze]
col_max_po    = [w['max_po']    for w in wiersze]

# Formatowanie Δ
col_delta_fmt = [
    f'+{d}' if d > 0 else str(d)
    for d in col_delta
]

# ── KOLORY KOMÓREK — TABELA 1 ─────────────────────────────────────────────────
delta_colors = [
    TURKUSOWY if d > 0 else CZERWONY
    for d in col_delta
]

n = len(wiersze)
row_fill = [
    '#EEF2F8' if i % 2 == 0 else BIALY
    for i in range(n)
]

# ── KOLORY KOMÓREK — TABELA 2 ─────────────────────────────────────────────────
# Min po — turkusowy jeśli najgorszy rok po reformach bije najlepszy rok przed
min_po_colors = [
    TURKUSOWY if mpo > mprzed else SZARY
    for mpo, mprzed in zip(col_min_po, col_max_przed)
]

# Max po — zawsze turkusowy (szczyt po reformach)
max_po_colors    = [TURKUSOWY] * n

# Min przed, Max przed — zawsze szary (okres historyczny)
min_przed_colors = [SZARY] * n
max_przed_colors = [SZARY] * n

# ══════════════════════════════════════════════════════════════════════════════
# TABELA 1 — Miary centralne i zmienność
# ══════════════════════════════════════════════════════════════════════════════
tabela1 = go.Figure(go.Table(
    columnwidth=[2.2, 1.2, 1.2, 1.4, 1.2, 1.2, 1.2, 1.2],
    header=dict(
        values=[
            '<b>Zmienna</b>',
            '<b>Śr.<br>przed</b>',
            '<b>Śr.<br>po</b>',
            '<b>Δ<br>średnia</b>',
            '<b>Med.<br>przed</b>',
            '<b>Med.<br>po</b>',
            '<b>Std<br>przed</b>',
            '<b>Std<br>po</b>',
        ],
        fill_color=NIEBIESKI,
        font=dict(color=BIALY, size=11, family='Arial'),
        align=['left'] + ['center'] * 7,
        height=40,
        line=dict(color=BIALY, width=1),
    ),
    cells=dict(
        values=[
            col_nazwa,
            col_sr_przed,
            col_sr_po,
            col_delta_fmt,
            col_med_przed,
            col_med_po,
            col_std_przed,
            col_std_po,
        ],
        fill_color=[
            row_fill,      # Zmienna
            row_fill,      # Śr. przed
            row_fill,      # Śr. po
            delta_colors,  # Δ — turkusowy/czerwony
            row_fill,      # Med. przed
            row_fill,      # Med. po
            row_fill,      # Std przed
            row_fill,      # Std po
        ],
        font=dict(
            color=[
                [NIEBIESKI] * n,  # Zmienna
                [SZARY] * n,      # Śr. przed
                [NIEBIESKI] * n,  # Śr. po
                [BIALY] * n,      # Δ — biały na kolorowym tle
                [SZARY] * n,      # Med. przed
                [NIEBIESKI] * n,  # Med. po
                [SZARY] * n,      # Std przed
                [NIEBIESKI] * n,  # Std po
            ],
            size=11,
            family='Arial',
        ),
        align=['left'] + ['center'] * 7,
        height=35,
        line=dict(color=SZARY_GRID, width=0.5),
    ),
))

tabela1.update_layout(
    title=dict(
        text=(
            '<b>Statystyki opisowe (1/2) · Miary centralne i zmienność (dane kwartalne 1999–2025)</b><br>'
            '<span style="font-size:11px;color:#8395A7">'
            'Źródła: Ministerstwo Finansów (Sprawozdania operatywne) · '
            'Eurostat (namq_10_gdp/CP_MNAC)<br>'
            'Opracowanie: Mateusz Durski'
            '</span>'
        ),
        x=0.01, xanchor='left',
        font=dict(size=15),
    ),
    height=340,
    paper_bgcolor=BG,
    margin=dict(t=110, b=10, l=20, r=20),
)

# ══════════════════════════════════════════════════════════════════════════════
# TABELA 2 — Rozstęp (Min / Max)
# ══════════════════════════════════════════════════════════════════════════════
tabela2 = go.Figure(go.Table(
    columnwidth=[2.2, 1.2, 1.2, 1.2, 1.2],
    header=dict(
        values=[
            '<b>Zmienna</b>',
            '<b>Min<br>przed</b>',
            '<b>Min<br>po</b>',
            '<b>Max<br>przed</b>',
            '<b>Max<br>po</b>',
        ],
        fill_color=NIEBIESKI,
        font=dict(color=BIALY, size=11, family='Arial'),
        align=['left'] + ['center'] * 4,
        height=40,
        line=dict(color=BIALY, width=1),
    ),
    cells=dict(
        values=[
            col_nazwa,
            col_min_przed,
            col_min_po,
            col_max_przed,
            col_max_po,
        ],
        fill_color=[
            row_fill,  # Zmienna
            row_fill,  # Min przed
            row_fill,  # Min po
            row_fill,  # Max przed
            row_fill,  # Max po
        ],
        font=dict(
            color=[
                [NIEBIESKI] * n,  # Zmienna
                min_przed_colors, # Min przed — szary
                min_po_colors,    # Min po — turkusowy jeśli > max przed
                max_przed_colors, # Max przed — szary
                max_po_colors,    # Max po — zawsze turkusowy
            ],
            size=11,
            family='Arial',
        ),
        align=['left'] + ['center'] * 4,
        height=35,
        line=dict(color=SZARY_GRID, width=0.5),
    ),
))

tabela2.update_layout(
    title=dict(
        text=(
            '<b>Statystyki opisowe (2/2) · Rozstęp (Min / Max) (dane kwartalne 1999–2025)</b><br>'
            '<span style="font-size:11px;color:#8395A7">'
            'Źródła: Ministerstwo Finansów (Sprawozdania operatywne) · '
            'Eurostat (namq_10_gdp/CP_MNAC)<br>'
            'Opracowanie: Mateusz Durski'
            '</span>'
        ),
        x=0.01, xanchor='left',
        font=dict(size=15),
    ),
    height=310,
    paper_bgcolor=BG,
    margin=dict(t=110, b=10, l=20, r=20),
)

# ── Export — TABELA 1 i TABELA 2 ─────────────────────────────────────────────────
# Ścieżka do pliku
file_path = OUT_DIR / "statystyki_opisowe.html"

# Zapis do HTML
with open(file_path, 'w', encoding='utf-8') as f:
    f.write('<html>\n<head><meta charset="utf-8"></head>\n<body>\n')
    
    # Dodaj nagłówek dla pierwszej tabeli
    f.write('<h2>Statystyki opisowe - dane kwartalne</h2>\n')
    f.write(tabela1.to_html(full_html=False, include_plotlyjs='cdn'))
    f.write('\n<br>\n')  # pusta linia między tabelami
    
    # Dodaj nagłówek dla drugiej tabeli
    f.write('<h2>Statystyki opisowe - dane kwartalne</h2>\n')
    f.write(tabela2.to_html(full_html=False, include_plotlyjs=False))
    f.write('\n</body>\n</html>')

print(f"✅ Zapisano: {file_path}")

# Wyświetlenie tabel w notebooku (opcjonalne)
tabela1.show()
tabela2.show()

✅ Zapisano: c:\Users\duros\OneDrive\Pulpit\Projekty Python\VAT\projekt_vat\outputs\html\statystyki_opisowe.html


In [8]:
del BG, BIALY, col_delta, col_delta_fmt, col_max_po, col_max_przed, col_med_po, col_med_przed, col_min_po, col_min_przed, col_nazwa, col_sr_po, col_sr_przed, col_std_po, col_std_przed, CZERWONY, decimals, delta_colors, f, kolumna, max_po_colors, max_przed_colors, min_po_colors, min_przed_colors, n, nazwa, NIEBIESKI, row_fill, s, SZARY, SZARY_GRID,tabela1, tabela2, TURKUSOWY, wiersze, ZMIENNE

In [9]:
# ── SEKCJA 4: HEATMAPA KORELACJI (dane kwartalne [RAW i DIFF]) ────────────────

# ── ZMIENNE ───────────────────────────────────────────────────────────────────
zmienne = [
    'vat_mld', 'cit_mld', 'akcyza_mld',
    'vat_pkb_pct', 'pkb_mld',
]

etykiety = {
    'vat_mld':     'VAT (mld)',
    'cit_mld':     'CIT (mld)',
    'akcyza_mld':  'Akcyza (mld)',
    'vat_pkb_pct': 'VAT/PKB %',
    'pkb_mld':     'PKB (mld)',
}
etykiety_lista = [etykiety[z] for z in zmienne]

# ── MACIERZE KORELACJI ────────────────────────────────────────────────────────
# Poziomy — pokazuje multikolinearność i wspólny trend nominalny
corr_poziomy = df[zmienne].corr(method='spearman').round(2)

# Zmiany r/r — eliminuje trend, pokazuje rzeczywiste współzależności
# pct_change() oblicza (t - t-1) / t-1, dropna() usuwa pierwszy wiersz NaN
df_diff   = df[zmienne].pct_change().dropna()
corr_diff = df_diff.corr(method='spearman').round(2)

# ── MASKOWANIE — tylko dolny trójkąt ─────────────────────────────────────────
def maskuj(corr):
    """
    Maskuje górny trójkąt macierzy przez NaN.
    k=1 zostawia przekątną (korelacja zmiennej z samą sobą = 1.0).
    """
    mask   = np.triu(np.ones(corr.shape, dtype=bool), k=1)
    corr_m = corr.copy()
    corr_m[mask] = np.nan
    return corr_m

corr_poziomy_m = maskuj(corr_poziomy)
corr_diff_m    = maskuj(corr_diff)

# ── KOLORY ────────────────────────────────────────────────────────────────────
CZERWONY   = '#C0392B'
NIEBIESKI  = '#1B3A6B'
BIALY      = '#FFFFFF'
BG         = '#F7F9FC'
SZARY_GRID = '#E4EAF2'

colorscale = [
    [0.0, CZERWONY],   # -1.0 — silna korelacja ujemna
    [0.5, BIALY],      #  0.0 — brak korelacji
    [1.0, NIEBIESKI],  # +1.0 — silna korelacja dodatnia
]

# ── FIGURA 1×2 ────────────────────────────────────────────────────────────────
heatmap = make_subplots(
    rows=1, cols=2,
    subplot_titles=(
        '① Korelacja poziomów',
        '② Korelacja zmian r/r',
    ),
    horizontal_spacing=0.12,
)

# ── HELPER — jeden trace heatmapy ─────────────────────────────────────────────
def dodaj_heatmape(heatmap, corr_m, row, col, showscale):
    """
    Rysuje heatmapę w subplocie.
    Jawne ustawienie xaxis/yaxis eliminuje błąd przypisania osi
    przy make_subplots z go.Heatmap.
    """
    # Mapowanie (row, col) → sufiks osi Plotly
    # (1,1) → '' | (1,2) → '2'
    os = '' if (row == 1 and col == 1) else '2'

    heatmap.add_trace(go.Heatmap(
        z=corr_m.values,
        x=etykiety_lista,
        y=etykiety_lista,
        colorscale=colorscale,
        zmin=-1, zmax=1,              # stała skala niezależna od danych
        text=corr_m.values,
        texttemplate='%{text:.2f}',
        textfont=dict(size=10, family='Arial'),
        hoverongaps=False,            # brak tooltipa dla NaN (górny trójkąt)
        hovertemplate=(
            '<b>%{y} × %{x}</b><br>'
            'Korelacja Spearmana: %{z:.2f}'
            '<extra></extra>'
        ),
        showscale=showscale,
        xaxis=f'x{os}',              # jawne przypisanie osi X do subplotu
        yaxis=f'y{os}',              # jawne przypisanie osi Y do subplotu
        colorbar=dict(
            title=dict(
                text='Spearman',
                font=dict(size=10, color='#5D6D7E'),
                side='right',
            ),
            tickvals=[-1, -0.5, 0, 0.5, 1],
            ticktext=['-1.0', '-0.5', '0.0', '+0.5', '+1.0'],
            tickfont=dict(size=9, color='#5D6D7E'),
            len=0.75,
            thickness=12,
            x=1.02,
        ),
    ), row=row, col=col)


# ── DODAJ OBA TRACE ───────────────────────────────────────────────────────────
dodaj_heatmape(heatmap, corr_poziomy_m, row=1, col=1, showscale=False)
dodaj_heatmape(heatmap, corr_diff_m,    row=1, col=2, showscale=True)

# ── PODTYTUŁY SUBPLOTÓW — rozszerzone o opis ─────────────────────────────────
for i, (tekst, kolor) in enumerate([
    ('Wspólny trend nominalny — multikolinearność', '#8395A7'),
    ('Po eliminacji trendu — rzeczywiste współzależności', '#8395A7'),
]):
    heatmap.layout.annotations[i].update(
        text=f'{heatmap.layout.annotations[i].text}<br>'
             f'<span style="font-size:10px;color:{kolor}">{tekst}</span>',
        font=dict(size=12),
    )

# ── LAYOUT ───────────────────────────────────────────────────────────────────
heatmap.update_layout(
    title=dict(
        text=(
            '<b>Heatmapa korelacji Spearmana · Poziomy vs Zmiany r/r</b><br>'
            '<span style="font-size:11px;color:#8395A7">'
            'Metoda: Spearman · Opracowanie: Mateusz Durski'
            '</span>'
        ),
        x=0.01, xanchor='left',
        font=dict(size=15),
    ),
    height=580,
    paper_bgcolor=BG,
    plot_bgcolor=BG,
    font=dict(family='Arial', size=10, color='#1C2833'),
    margin=dict(t=140, b=80, l=100, r=100),
)

# ── OSIE ─────────────────────────────────────────────────────────────────────
for col in [1, 2]:
    heatmap.update_xaxes(
        tickangle=-45,
        tickfont=dict(size=10),
        showgrid=False,
        row=1, col=col,
    )
    heatmap.update_yaxes(
        tickfont=dict(size=10),
        showgrid=False,
        autorange='reversed',   # pierwsza zmienna na górze
        row=1, col=col,
    )


# ── EXPORT ───────────────────────────────────────────────────────────────────
file_path = OUT_DIR / "heatmapa_korelacji.html"

heatmap.write_html(
    file_path,
    include_plotlyjs="cdn",
)

print(f"✅ Zapisano: {file_path}")

heatmap.show()

✅ Zapisano: c:\Users\duros\OneDrive\Pulpit\Projekty Python\VAT\projekt_vat\outputs\html\heatmapa_korelacji.html


In [10]:
del BG, BIALY, col, colorscale, corr_diff, corr_diff_m, corr_poziomy, corr_poziomy_m, CZERWONY, etykiety, etykiety_lista, heatmap, i, kolor, NIEBIESKI, SZARY_GRID, tekst, zmienne

In [11]:
# ── SEKCJA 5: SCATTERPLOT (dane roczne ) ──────────────────────────────────────


# ── DANE ──────────────────────────────────────────────────────────────────────
przed = df_rok[df_rok.index <= 2015]
po    = df_rok[df_rok.index >= 2016]

def linia_trendu(x, y):
    """
    Oblicza współczynniki regresji liniowej y = a*x + b.
    Zwraca wartości y dla zakresu x (do rysowania linii).
    """
    a, b    = np.polyfit(x, y, deg=1)
    x_range = np.linspace(x.min(), x.max(), 100)
    y_range = a * x_range + b
    return x_range, y_range, a, b

x_przed, y_przed_trend, a_przed, b_przed = linia_trendu(
    przed['pkb_mld'].values,
    przed['vat_mld'].values,
)
x_po, y_po_trend, a_po, b_po = linia_trendu(
    po['pkb_mld'].values,
    po['vat_mld'].values,
)

# ── KOLORY ────────────────────────────────────────────────────────────────────
CZERWONY    = '#C0392B'
NIEBIESKI   = '#1B3A6B'
SZARY_PRZED = '#A9B8C8'
SZARY_GRID  = '#E4EAF2'
BG          = '#F7F9FC'
BIALY       = '#FFFFFF'

# ── FIGURA ────────────────────────────────────────────────────────────────────
scatter = go.Figure()

# ── PUNKTY PRZED 2016 ─────────────────────────────────────────────────────────
scatter.add_trace(go.Scatter(
    x=przed['pkb_mld'],
    y=przed['vat_mld'],
    mode='markers+text',
    name='Przed 2016 (2003–2015)',
    marker=dict(
        color=SZARY_PRZED,
        size=10,
        line=dict(color=BIALY, width=1),
        opacity=0.85,
    ),
    text=przed.index.astype(str),
    textposition='top center',
    textfont=dict(size=9, color=SZARY_PRZED),
    hovertemplate=(
        '<b>%{text}</b><br>'
        'PKB: %{x:.1f} mld PLN<br>'
        'VAT: %{y:.1f} mld PLN'
        '<extra></extra>'
    ),
))

# ── PUNKTY PO 2016 ────────────────────────────────────────────────────────────
scatter.add_trace(go.Scatter(
    x=po['pkb_mld'],
    y=po['vat_mld'],
    mode='markers+text',
    name='Po 2016 (2016–2024)',
    marker=dict(
        color=CZERWONY,
        size=10,
        line=dict(color=BIALY, width=1),
        opacity=0.85,
    ),
    text=po.index.astype(str),
    textposition='top center',
    textfont=dict(size=9, color=CZERWONY),
    hovertemplate=(
        '<b>%{text}</b><br>'
        'PKB: %{x:.1f} mld PLN<br>'
        'VAT: %{y:.1f} mld PLN'
        '<extra></extra>'
    ),
))

# ── LINIA TRENDU PRZED 2016 ───────────────────────────────────────────────────
scatter.add_trace(go.Scatter(
    x=x_przed,
    y=y_przed_trend,
    mode='lines',
    name=f'Trend przed 2016 (nachylenie: {a_przed:.3f})',
    line=dict(color=SZARY_PRZED, width=2, dash='dash'),
    hoverinfo='skip',
))

# ── LINIA TRENDU PO 2016 ──────────────────────────────────────────────────────
scatter.add_trace(go.Scatter(
    x=x_po,
    y=y_po_trend,
    mode='lines',
    name=f'Trend po 2016 (nachylenie: {a_po:.3f})',
    line=dict(color=CZERWONY, width=2, dash='dash'),
    hoverinfo='skip',
))

# ── ADNOTACJE — nachylenia linii trendu ───────────────────────────────────────
scatter.add_annotation(
    x=x_przed[-1], y=y_przed_trend[-1],
    text=f'<b>{a_przed:.3f} mld VAT<br>/ mld PKB</b>',
    showarrow=True,
    arrowhead=2, arrowcolor=SZARY_PRZED, arrowwidth=1.2,
    ax=40, ay=-30,
    font=dict(size=9, color=SZARY_PRZED),
)

scatter.add_annotation(
    x=x_po[-1], y=y_po_trend[-1],
    text=f'<b>{a_po:.3f} mld VAT<br>/ mld PKB</b>',
    showarrow=True,
    arrowhead=2, arrowcolor=CZERWONY, arrowwidth=1.2,
    ax=40, ay=-30,
    font=dict(size=9, color=CZERWONY),
)

# ── LAYOUT ───────────────────────────────────────────────────────────────────
scatter.update_layout(
    title=dict(
        text=(
            '<b>Scatter · VAT vs PKB — zmiana relacji przed i po reformach 2016</b><br>'
            '<span style="font-size:11px;color:#8395A7">'
            'Nachylenie linii trendu = mld PLN VAT na 1 mld PLN PKB<br>'
            'Opracowanie: Mateusz Durski'
            '</span>'
        ),
        x=0.01, xanchor='left',
        font=dict(size=15),
    ),
    height=580,
    paper_bgcolor=BG,
    plot_bgcolor=BIALY,
    hovermode='closest',
    legend=dict(
        orientation='h',
        y=-0.12, x=0.5, xanchor='center',
        font=dict(size=10),
        bgcolor='rgba(247,249,252,0.9)',
        bordercolor=SZARY_GRID, borderwidth=1,
    ),
    xaxis=dict(
        title=dict(text='PKB (mld PLN)', font=dict(size=11)),
        showgrid=True, gridcolor=SZARY_GRID,
        zeroline=False,
        tickfont=dict(size=10),
    ),
    yaxis=dict(
        title=dict(text='VAT (mld PLN)', font=dict(size=11)),
        showgrid=True, gridcolor=SZARY_GRID,
        zeroline=False,
        tickfont=dict(size=10),
    ),
    margin=dict(t=120, b=80, l=80, r=60),
)

# ── EXPORT ───────────────────────────────────────────────────────────────────
file_path = OUT_DIR / "scatter_vat_pkb.html"

scatter.write_html(
    file_path,
    include_plotlyjs="cdn",
)

print(f"✅ Zapisano: {file_path}")

scatter.show()

✅ Zapisano: c:\Users\duros\OneDrive\Pulpit\Projekty Python\VAT\projekt_vat\outputs\html\scatter_vat_pkb.html


In [12]:
del a_po, b_przed, a_przed, b_po, BG, BIALY, CZERWONY, NIEBIESKI, po, przed, scatter, SZARY_GRID, SZARY_PRZED, x_po, x_przed, y_po_trend, y_przed_trend